# 01 — The Scaling Problem

**Engineering question**

Why does simply adding more quantum devices stop being an effective scaling strategy?

This notebook develops the engineering motivation behind integrated quantum photonics:

```text
Need more channels
→ Replicate devices
→ Packaging complexity
→ Calibration burden
→ Compound yield
→ Multiplexing becomes preferable
```

Notebook 00 established the repo roadmap.  
Notebook 01 explains why the roadmap needs a shift from **replication** to **multiplexing**.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/01_outputs.zip
```

The final cell supports both:

- **Google Colab**: starts a real browser download with `files.download(...)`
- **Jupyter**: displays a clickable `FileLink`


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Compound yield

If each replicated device has yield \(\eta\), then system yield decays as:

```text
Y = ηᴺ
```

The key point is not the exact value of \(\eta\).  
The key point is that replication multiplies independent failure probabilities.


In [ ]:
N = np.arange(0, 501)
etas = [0.999, 0.99, 0.98, 0.95]

fig, ax = plt.subplots(figsize=(8, 4.8))

for eta in etas:
    ax.plot(N, eta ** N, linewidth=2.3, label=f"η = {eta}")

ax.set_title("Compound Yield Under Device Replication", fontsize=16)
ax.set_xlabel("Number of replicated devices")
ax.set_ylabel("System yield")
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
ax.legend()

for n in [10, 100, 300]:
    ax.axvline(n, linewidth=0.8, alpha=0.25)

savefig(fig, "01_compound_yield.png")
plt.show()


## 2. Packaging growth

Replication adds channels, but it also adds:

- optical paths,
- electrical lines,
- thermal tuning,
- alignment needs,
- calibration degrees of freedom.

The model below is deliberately simple: packaging burden grows faster than device count.


In [ ]:
devices = np.arange(1, 65)
linear_channels = devices
packaging_burden = devices ** 1.45

fig, ax = plt.subplots(figsize=(8, 4.8))

ax.plot(devices, linear_channels, linewidth=2.5, label="Useful channels: ~N")
ax.plot(devices, packaging_burden, linewidth=2.5, label="Packaging burden: faster than N")

ax.set_title("Packaging Burden Can Outrun Channel Count", fontsize=16)
ax.set_xlabel("Replicated devices")
ax.set_ylabel("Relative scale")
ax.grid(True, alpha=0.3)
ax.legend()

savefig(fig, "01_packaging_growth.png")
plt.show()


## 3. Replication stack

A replication strategy says:

```text
N channels
→ N sources
→ N detectors
→ N optical paths
→ N calibration targets
```

This is the architecture that frequency multiplexing tries to avoid.


In [ ]:
levels = [
    "N channels",
    "N sources",
    "N detectors",
    "N optical paths",
    "N calibration targets",
]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
y = np.arange(len(levels))[::-1]

for yi, label in zip(y, levels):
    box = plt.Rectangle((0.15, yi - 0.28), 0.7, 0.56, fill=False, linewidth=2)
    ax.add_patch(box)
    ax.text(0.5, yi, label, ha="center", va="center", fontsize=12)

for yi in y[:-1]:
    ax.annotate("", xy=(0.5, yi - 0.7), xytext=(0.5, yi - 0.3), arrowprops=dict(arrowstyle="->", linewidth=2))

ax.set_title("Replication Stack: Channels Bring Hardware With Them", fontsize=16)
ax.set_xlim(0, 1)
ax.set_ylim(-0.8, len(levels) - 0.2)
ax.axis("off")

savefig(fig, "01_replication_stack.png")
plt.show()


## 4. Replication versus modal scaling

Device replication asks for more physical objects.

Modal scaling asks whether a richer optical system can support many quantum channels inside fewer physical devices.

This is the motivation for Notebook 05: frequency multiplexing.


In [ ]:
effort = np.arange(1, 17)

replication = effort
modal_capacity = np.array([1, 2, 3, 5, 8, 12, 17, 24, 32, 45, 60, 75, 95, 120, 145, 170])

fig, ax = plt.subplots(figsize=(8.5, 5))

ax.plot(effort, replication, "o-", linewidth=2.7, label="Replication: one device → one channel")
ax.plot(effort, modal_capacity, "s-", linewidth=2.7, label="Modal scaling: many modes per device")

ax.set_title("Replication Versus Modal Scaling", fontsize=16)
ax.set_xlabel("Engineering effort / design generation")
ax.set_ylabel("Available optical quantum channels")
ax.grid(True, alpha=0.3)
ax.legend()

savefig(fig, "01_replication_vs_modes.png")
plt.show()


## 5. Scaling bottlenecks

Replication does not fail because one component is impossible.

It fails because many constraints accumulate:

```text
fabrication
→ packaging
→ alignment
→ calibration
→ loss
→ yield
```

Each stage consumes engineering margin.


In [ ]:
labels = ["Fabrication", "Packaging", "Alignment", "Calibration", "Loss", "Yield"]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(11, 3.0))

ax.scatter(x, np.zeros_like(x), s=300, zorder=3)
for i in range(len(labels) - 1):
    ax.annotate("", xy=(i + 1, 0), xytext=(i, 0), arrowprops=dict(arrowstyle="->", linewidth=2))

for xi, label in zip(x, labels):
    ax.text(xi, -0.2, label, ha="center", va="top", fontsize=11)

ax.set_title("Engineering Bottlenecks Under Replication", fontsize=16)
ax.set_xlim(-0.4, len(labels) - 0.6)
ax.set_ylim(-0.55, 0.35)
ax.axis("off")

savefig(fig, "01_scaling_bottlenecks.png")
plt.show()


## 6. Transition to multiplexing

At small scale, replication is intuitive.

At larger scale, a different strategy can become more attractive:

```text
fewer devices
more modes
more integrated routing
shared measurement architecture
```

This figure is conceptual: the crossing point marks where multiplexing becomes the better engineering question.


In [ ]:
scale = np.linspace(0, 10, 300)

replication_cost = 0.08 * scale**2 + 0.3 * scale
multiplexing_cost = 0.55 * scale + 1.5

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(scale, replication_cost, linewidth=2.8, label="Replication burden")
ax.plot(scale, multiplexing_cost, linewidth=2.8, label="Multiplexing burden")

cross_idx = np.argmin(np.abs(replication_cost - multiplexing_cost))
cross_x = scale[cross_idx]
cross_y = replication_cost[cross_idx]

ax.scatter([cross_x], [cross_y], s=80, zorder=4)
ax.text(cross_x + 0.25, cross_y, "transition", va="center")

ax.set_title("Transition Toward Multiplexing", fontsize=16)
ax.set_xlabel("System scale")
ax.set_ylabel("Relative engineering burden")
ax.grid(True, alpha=0.3)
ax.legend()

savefig(fig, "01_transition_to_multiplexing.png")
plt.show()


## 7. Summary table

Notebook 01 exists to justify the next move:

```text
replication becomes burdened
→ frequency multiplexing becomes the next engineering question
```


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Constraint": "Replication",
            "Engineering effect": "More physical hardware",
            "Why it matters": "Each channel brings components with it.",
            "Next question": "Can one device support many channels?",
            "Next notebook": "05_frequency_multiplexing",
        },
        {
            "Constraint": "Packaging",
            "Engineering effect": "More interconnects and optical paths",
            "Why it matters": "Physical organization becomes a scaling problem.",
            "Next question": "Can channels be carried by optical modes?",
            "Next notebook": "05_frequency_multiplexing",
        },
        {
            "Constraint": "Calibration",
            "Engineering effect": "More tuning targets",
            "Why it matters": "Large systems accumulate control requirements.",
            "Next question": "Can resonators organize many modes coherently?",
            "Next notebook": "07_quantum_frequency_combs",
        },
        {
            "Constraint": "Yield",
            "Engineering effect": "Compound decay",
            "Why it matters": "Independent component imperfections multiply.",
            "Next question": "Can modal resources reduce replicated components?",
            "Next notebook": "05_frequency_multiplexing",
        },
        {
            "Constraint": "Transition",
            "Engineering effect": "Multiplexing becomes preferable",
            "Why it matters": "The system shifts from counting devices to managing modes.",
            "Next question": "How does frequency multiplexing work?",
            "Next notebook": "05_frequency_multiplexing",
        },
    ]
)

summary.to_csv(CSV / "01_summary.csv", index=False)
summary.to_json(JS / "01_summary.json", orient="records", indent=2)

summary


## 8. Export and download

This cell packages all outputs and starts a download.

It uses Colab's native downloader when available.  
For local Jupyter, it falls back to a clickable `FileLink`.


In [ ]:
zip_path = RES / "01_outputs.zip"

files_to_zip = [
    FIG / "01_compound_yield.png",
    FIG / "01_packaging_growth.png",
    FIG / "01_replication_stack.png",
    FIG / "01_replication_vs_modes.png",
    FIG / "01_scaling_bottlenecks.png",
    FIG / "01_transition_to_multiplexing.png",
    CSV / "01_summary.csv",
    JS / "01_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Next notebook

**05 — Frequency Multiplexing**

Notebook 01 shows why replication becomes constrained.

Notebook 05 asks the natural next question:

```text
If adding more devices becomes expensive,
how can one optical system carry many channels?
```

Candidate figures:

```text
05_wavelength_division_multiplexing.png
05_frequency_ladder.png
05_modes_as_channels.png
05_one_device_many_modes.png
```
